# 05 Safety, Approval, and Risk Controls (OpenClaw, 2026)

## What This Lesson Is
Model risk controls by classifying actions and requiring approvals for higher-risk operations.

## Scientific Lens
- Concept: Risk-tiered control gates for agent-invoked operations.
- Measure: False-negative rate for deterministic risk classifier.
- Validity Limit: Rule-based classification can miss nuanced attack patterns.


## How It Works
1. Score planned actions with deterministic risk rules.
2. Require explicit approvals for medium/high-risk operations.
3. Inspect live configuration state to verify policy surface before execution.


In [ ]:
import os
import shutil

HAS_OPENCLAW = shutil.which("openclaw") is not None
print("openclaw available:", HAS_OPENCLAW)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL", "<unset>"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: runs real OpenClaw CLI operations when available; otherwise prints explicit skip guidance.


In [ ]:
# Deterministic Demo
actions = [
    {"name": "read docs", "touches_secrets": False, "writes_repo": False},
    {"name": "edit deployment", "touches_secrets": False, "writes_repo": True},
    {"name": "rotate token", "touches_secrets": True, "writes_repo": True},
]
def classify(action):
    if action["touches_secrets"]:
        return "high"
    if action["writes_repo"]:
        return "medium"
    return "low"
risk = {a["name"]: classify(a) for a in actions}
approval_required = {k: v in {"medium", "high"} for k, v in risk.items()}
print(risk)
print(approval_required)
assert approval_required["read docs"] is False
assert approval_required["rotate token"] is True


In [ ]:
# Live Demo
import shutil
import subprocess

if shutil.which("openclaw") is None:
    print("Skipping live safety demo: openclaw CLI is not installed.")
else:
    cmd = ["openclaw", "config", "get"]
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    raw = (proc.stdout or proc.stderr).strip()
    redacted = raw.replace("apiKey", "apiKey(REDACTED_KEY_NAME)")
    print(redacted[:1500])


## Applied Labs
1. Add `network_scope` attribute and classify outbound write actions as high risk.
2. Emit approval ticket payload for medium/high-risk actions.
3. Simulate bypass attempt and verify classifier still marks it high risk.

## Validation Checklist
- Risk logic is transparent and tied to concrete attributes.
- Approval requirements are derived, not hard-coded per action name.
- Live config review avoids exposing raw secrets in output.

## Further Reading
- OWASP LLM Top 10: https://owasp.org/www-project-top-10-for-large-language-model-applications/
- NIST AI RMF playbook: https://airc.nist.gov/AI_RMF_Knowledge_Base/Playbook
- OpenClaw docs: https://docs.openclaw.ai
